# Chapitre 3 : Ingestion et Export multi-formats (I/O)
## Au-delà du simple `read_csv` : les réalités du Data Engineer

Dans un pipeline de données, la phase **Extract (E)** et **Load (L)** repose sur la capacité à communiquer avec des systèmes hétérogènes. Deux formats dominent le monde réel :
1. **Les formats textuels / d'échange (CSV, JSON, Excel)** : universels et lisibles par un humain, mais lourds, non typés et fragiles (séparateurs, encodages).
2. **Les formats colonnaires binaires (Parquet)** : le standard moderne de stockage analytique (compressé, typage préservé, lecture ultra-rapide par projection de colonnes).

Dans ce chapitre, nous allons voir comment dompter les pièges d'ingestion courants et mesurer l'impact colossal du choix d'un format.

In [ ]:
import os
import time
import pandas as pd
import numpy as np

# Création d'un dossier de travail pour nos fichiers
os.makedirs("data_sandbox", exist_ok=True)
print("Dossier 'data_sandbox' prêt !")

## 1. Le CSV et ses chausse-trapes (Séparateurs, Encodages, Dates)
Exécutez la cellule suivante pour générer un fichier `ventes_raw.csv` délibérément représentatif des exports sales rencontrés en entreprise (séparateur point-virgule, accents encodés en `latin1`, dates sous forme de texte).

In [ ]:
# Génération du fichier piégé
csv_content = (
    "id_commande;date_achat;client;total_ttc;ville\n"
    "CMD-01;2024-03-01 10:15:00;Élodie;120.50;Nîmes\n"
    "CMD-02;2024-03-01 11:30:00;François;45.00;Béthune\n"
    "CMD-03;2024-03-02 09:05:00;Hélène;310.20;Alès\n"
    "CMD-04;2024-03-02 14:45:00;Jérôme;89.90;Saint-Étienne\n"
)

with open("data_sandbox/ventes_raw.csv", "w", encoding="latin1") as f:
    f.write(csv_content)

print("Fichier 'data_sandbox/ventes_raw.csv' généré avec succès en latin1 !")

### 🧪 Exercice 1 : Ingestion robuste du CSV
Si vous faites un simple `pd.read_csv('data_sandbox/ventes_raw.csv')`, l'opération échouera ou produira un DataFrame à 1 seule colonne mal encodée.

**Objectifs :**
1. Chargez le fichier dans `df_ventes` en spécifiant le bon séparateur `sep`.
2. Définissez le bon encodage (`encoding='latin1'`).
3. Demandez à Pandas de convertir immédiatement `date_achat` en véritable objet `datetime` dès la lecture via le paramètre `parse_dates`.

In [ ]:
# TODO : Complétez les paramètres de pd.read_csv
df_ventes = pd.read_csv(
    "data_sandbox/ventes_raw.csv",
    sep=...,
    encoding=...,
    parse_dates=...
)

df_ventes.info()

In [ ]:
assert df_ventes.shape == (4, 5), f"Attendu (4, 5), obtenu {df_ventes.shape}. Vérifiez le séparateur !"
assert pd.api.types.is_datetime64_any_dtype(df_ventes["date_achat"]), "date_achat doit être de type datetime64 !"
assert df_ventes.loc[0, "client"] == "Élodie", "Problème d'encodage : Élodie est mal décodée."
print("✅ Exercice 1 validé avec succès !")

## 2. Le format JSON et les flux d'API (Records vs JSON Lines)
Dans les architectures modernes (Microservices, Kafka, API REST), la donnée circule souvent en JSON sous deux formes principales :
- **Format standard / records** : Une liste d'objets `[ {...}, {...} ]`.
- **JSON Lines (`.jsonl`)** : Un objet JSON indépendant par ligne (très utilisé pour le streaming et les logs).

Exécutez la cellule ci-dessous pour générer un fichier de logs serveur au format **JSON Lines** :

In [ ]:
jsonl_content = (
    '{"timestamp": 1710000000, "ip": "192.168.1.10", "status": 200, "endpoint": "/api/v1/products"}\n'
    '{"timestamp": 1710000015, "ip": "192.168.1.42", "status": 404, "endpoint": "/api/v1/admin"}\n'
    '{"timestamp": 1710000030, "ip": "10.0.0.5", "status": 500, "endpoint": "/api/v1/checkout"}\n'
    '{"timestamp": 1710000045, "ip": "192.168.1.10", "status": 200, "endpoint": "/api/v1/products"}\n'
)

with open("data_sandbox/server_logs.jsonl", "w", encoding="utf-8") as f:
    f.write(jsonl_content)

print("Fichier 'data_sandbox/server_logs.jsonl' créé !")

### 🧪 Exercice 2 : Ingestion de JSON Lines
Consultez l'aide de `pd.read_json` (paramètre `lines=True`).

**Objectifs :**
1. Chargez le fichier `data_sandbox/server_logs.jsonl` dans `df_logs`.
2. Convertissez la colonne `timestamp` (exprimée en secondes Unix) en vraie date lisible via `pd.to_datetime(..., unit='s')`.

In [ ]:
# TODO : Chargement et conversion de timestamp
df_logs = pd.read_json("data_sandbox/server_logs.jsonl", lines=...)
df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"], unit=...)

df_logs.head()

In [ ]:
assert df_logs.shape == (4, 4), "Erreur sur le chargement du JSONL."
assert pd.api.types.is_datetime64_any_dtype(df_logs["timestamp"]), "Le timestamp doit être converti en datetime."
assert df_logs["status"].tolist() == [200, 404, 500, 200], "Les statuts HTTP ne correspondent pas."
print("✅ Exercice 2 validé avec succès !")

## 3. Le duel : CSV vs Parquet (Performance & Stockage)
Pourquoi Apache Parquet est-il devenu la norme absolue en Data Engineering ?
- **Format colonnaire** : On ne lit que les colonnes nécessaires sans scanner toute la ligne.
- **Compression native** (généralement `snappy` ou `gzip`).
- **Préservation du schéma** : Si une colonne est `int32`, `datetime` ou `category`, elle reste intacte sans ré-inférence à la relecture.

Mettons cela en évidence avec un benchmark en conditions réelles sur un volume de 200 000 lignes !

In [ ]:
print("Génération d'un jeu de données de test (200 000 lignes)...")
np.random.seed(42)
n_rows = 200_000

df_bench = pd.DataFrame({
    "id": np.arange(n_rows),
    "categorie": np.random.choice(["Électronique", "Maison", "Mode", "Jardin", "Auto"], size=n_rows),
    "prix_unitaire": np.random.uniform(5.0, 500.0, size=n_rows).round(2),
    "quantite": np.random.randint(1, 10, size=n_rows),
    "date_vente": pd.date_range("2024-01-01", periods=n_rows, freq="s")
})
print("Jeu de données prêt en mémoire :", df_bench.shape)

### 🧪 Exercice 3 : Sauvegarde et Benchmark comparatif

**Objectifs :**
1. Sauvegardez `df_bench` au format CSV (`data_sandbox/benchmark.csv`) sans exporter l'index (`index=False`).
2. Sauvegardez `df_bench` au format Parquet (`data_sandbox/benchmark.parquet`).
3. Exécutez le script d'audit fourni pour comparer la taille des fichiers et la vitesse de relecture.

In [ ]:
# TODO : Exportez les fichiers
# 1. Export CSV sans l'index
df_bench.to_csv("data_sandbox/benchmark.csv", index=...)

# 2. Export Parquet
df_bench.to_parquet("data_sandbox/benchmark.parquet")

print("Exports terminés !")

In [ ]:
# Comparaison des tailles sur disque
size_csv_mb = os.path.getsize("data_sandbox/benchmark.csv") / (1024 * 1024)
size_parquet_mb = os.path.getsize("data_sandbox/benchmark.parquet") / (1024 * 1024)

# Benchmark de temps de lecture
t0 = time.time()
_ = pd.read_csv("data_sandbox/benchmark.csv")
time_csv = time.time() - t0

t0 = time.time()
_ = pd.read_parquet("data_sandbox/benchmark.parquet")
time_parquet = time.time() - t0

print(f"--- RÉSULTATS DU BENCHMARK ---")
print(f"Taille sur disque : CSV = {size_csv_mb:.2f} Mo | Parquet = {size_parquet_mb:.2f} Mo  (Ratio: x{size_csv_mb/size_parquet_mb:.1f})")
print(f"Temps de lecture  : CSV = {time_csv:.3f} s  | Parquet = {time_parquet:.3f} s  (Vitesse: x{time_csv/time_parquet:.1f})")

assert size_parquet_mb < size_csv_mb, "Le fichier Parquet doit être plus léger que le CSV !"
assert time_parquet < time_csv, "La lecture Parquet doit être plus rapide que le CSV !"
print("\n🚀 Démonstration confirmée : Parquet surpasse le CSV en taille et en vitesse !")

---
# 🏆 Mini-Challenge Final : Le Pipeline d'Ingestion Multi-Sources

### Énoncé du problème :
Vous devez concevoir la première brique d'un pipeline ETL. Deux services vous transmettent des données :
1. Le service logistique envoie un CSV (`data_sandbox/flux_a.csv`).
2. Le service commercial envoie un JSON standard (`data_sandbox/flux_b.json`).

Exécutez la cellule ci-dessous pour créer les deux sources :

In [ ]:
# Création flux A (CSV délimité par des pipes '|')
with open("data_sandbox/flux_a.csv", "w", encoding="utf-8") as f:
    f.write("ref_article|entrepot|stock_disponible\nART-10;Nord;450\nART-20;Sud;120\nART-30;Est;0\n")
    
# Création flux B (JSON)
with open("data_sandbox/flux_b.json", "w", encoding="utf-8") as f:
    f.write(
        '[{"ref_article": "ART-40", "entrepot": "Ouest", "stock_disponible": 230},'
        '{"ref_article": "ART-50", "entrepot": "Nord", "stock_disponible": 85}]'
    )

print("Sources A et B créées !")

### Vos consignes :
1. Chargez `flux_a.csv` (attention au séparateur `|`) dans `df_a`.
2. Chargez `flux_b.json` dans `df_b`.
3. Concaténez ces deux DataFrames verticalement avec `pd.concat([df_a, df_b], ignore_index=True)` dans `df_consolidated`.
4. Filtrez pour ne garder que les articles ayant un stock strictement supérieur à 0 (`stock_disponible > 0`).
5. Exportez le résultat filtré final sous le nom `data_sandbox/stocks_propres.parquet`.

In [ ]:
# TODO : Votre code ici
df_a = ...
df_b = ...

df_consolidated = ...
df_final = ...

# Export Parquet
...

In [ ]:
# Vérification automatique
assert os.path.exists("data_sandbox/stocks_propres.parquet"), "Le fichier stocks_propres.parquet n'a pas été trouvé !"

df_check = pd.read_parquet("data_sandbox/stocks_propres.parquet")
assert len(df_check) == 4, f"Attendu 4 articles en stock > 0, trouvé : {len(df_check)}"
assert "ART-30" not in df_check["ref_article"].values, "L'article ART-30 avec stock 0 n'aurait pas dû être conservé."
assert list(df_check.columns) == ["ref_article", "entrepot", "stock_disponible"], "Les colonnes sont incorrectes."

print("🎉 FÉLICITATIONS ! Vous avez brillamment validé le Chapitre 3 (Ingestion & Export) !")